# Thư viện và Dữ liệu

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [ ]:
# tải dữ liệu
# Mỗi hàng = 1 khách hàng × 1 tháng
# Nhãn y = 1 nếu khách mua trong tháng đó, 0 nếu không
df = pd.read_csv('../tnbike_data_customers.csv', index_col=0, parse_dates=True)
df.head()

In [ ]:
# đổi tên biến
df = df.rename(columns={'bought': 'y'})
df

# Chuẩn bị cho GBM Classifier

In [ ]:
# đặc trưng (features) và nhãn (label)
feature_cols = [c for c in df.columns if c != 'y']
X = df[feature_cols]
y = df['y']
print(f"Features: {feature_cols}")
print(f"Tỉ lệ mua hàng: {y.mean():.2%}")

In [ ]:
# tạo đối tượng chuỗi thời gian cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=False)

In [ ]:
# scale dữ liệu
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# scale nhãn (giữ nguyên)
print(f"X shape: {X_scaled.shape}")
print(f"y shape: {y.shape}")

# Mô hình GBM Classifier

In [ ]:
# mô hình cơ sở
# https://xgboost.readthedocs.io/en/stable/python/python_api.html
model = XGBClassifier(n_estimators=100,
                      max_depth=4,
                      learning_rate=0.1,
                      subsample=0.8,
                      colsample_bytree=0.8,
                      use_label_encoder=False,
                      eval_metric='logloss',
                      random_state=42)

In [ ]:
# kiểm định chéo
cv_scores = cross_val_score(model, X_scaled, y,
                            cv=cv,
                            scoring='roc_auc',
                            verbose=2)
cv_scores

In [ ]:
# hiệu suất CV
print(f"AUC-ROC trung bình: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Tinh chỉnh tham số

In [ ]:
# lưới tham số
param_grid = {'n_estimators':    [100, 200],
              'max_depth':        [3, 4, 5],
              'learning_rate':    [0.05, 0.1],
              'subsample':        [0.7, 0.9],
              'colsample_bytree': [0.7, 0.9]}
grid = ParameterGrid(param_grid)
len(list(grid))

In [ ]:
# vòng lặp tinh chỉnh tham số
auc_scores = []
i = 1
# duyệt từng bộ tham số
for params in grid:
    print(f"{i} / {len(list(grid))}")
    # mô hình
    model = XGBClassifier(**params,
                          use_label_encoder=False,
                          eval_metric='logloss',
                          random_state=42)
    # CV
    cv_scores = cross_val_score(model, X_scaled, y,
                                cv=cv,
                                scoring='roc_auc',
                                verbose=0)
    # đo và lưu sai số
    error = cv_scores.mean()
    auc_scores.append(error)
    i += 1

In [ ]:
tuning_results = pd.DataFrame(grid)
tuning_results['auc'] = auc_scores
tuning_results

In [ ]:
# xuất tham số tốt nhất
best_params = tuning_results[tuning_results.auc == tuning_results.auc.max()].transpose()
best_params.to_csv("../Forecasting Product/best_params_classifier.csv")
best_params